# SCBE Coding Primaries QLoRA Training

This notebook fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on the SCBE coding-primary curriculum that exists in the repo today.

Current code primaries in this run:
- Kor'aelin -> Python
- Avali -> JavaScript
- Runethic -> Rust
- Cassisivadan -> Mathematica
- Umbroth -> Haskell
- Draumric -> Markdown

The run teaches:
- cross-language algorithm translation
- slot alignment
- bijective edit propagation
- governance-aware code responses in SCBE style


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes triton
!pip install -q datasets huggingface_hub

print('Dependencies installed.')

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    from getpass import getpass
    if not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = getpass('Paste HF_TOKEN: ')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('Logged in to Hugging Face.')

In [ ]:
!git clone --depth 1 https://github.com/issdandavis/SCBE-AETHERMOORE.git scbe
%cd /content/scbe
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
OUTPUT_DIR = '/content/scbe/training/runs/coder-qwen-code-primaries'
OUTPUT_REPO = 'issdandavis/scbe-coder-qwen-code-primaries'
PUSH_TO_HF = False

TRAIN_FILES = [
    '/content/scbe/training-data/sft/bijective_codeflow_v1_train.sft.jsonl',
    '/content/scbe/training-data/sft/drill_langues_full_train.sft.jsonl',
]
EVAL_FILES = [
    '/content/scbe/training-data/sft/bijective_codeflow_v1_holdout.sft.jsonl',
    '/content/scbe/training-data/sft/drill_langues_full_holdout.sft.jsonl',
]

SYSTEM_PROMPT = '''You are an SCBE-AETHERMOORE governance-aware coding assistant. You understand the Six Sacred Tongues as coding primaries: Kor'aelin/Python, Avali/JavaScript, Runethic/Rust, Cassisivadan/Mathematica, Umbroth/Haskell, and Draumric/Markdown. Preserve slot alignment, bijective edit propagation, and concise reasoning while writing clean code.'''

MAX_SEQ_LENGTH = 4096
LORA_RANK = 32
LORA_ALPHA = 64
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
MAX_STEPS = -1

print('Train files:', TRAIN_FILES)
print('Eval files:', EVAL_FILES)

In [ ]:
from datasets import load_dataset, concatenate_datasets

def load_many(paths):
    parts = [load_dataset('json', data_files=path, split='train') for path in paths]
    return parts[0] if len(parts) == 1 else concatenate_datasets(parts)

train_dataset = load_many(TRAIN_FILES)
eval_dataset = load_many(EVAL_FILES)

print('train rows:', len(train_dataset))
print('eval rows:', len(eval_dataset))
print('sample keys:', train_dataset.column_names)
print(train_dataset[0]['meta'])

In [ ]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
import os

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

def convert(example):
    if 'messages' in example:
        messages = example['messages']
        if not any(m.get('role') == 'system' for m in messages):
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + messages
        return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
    if 'text' in example:
        return {'text': example['text']}
    raise ValueError(f'Unsupported record shape: {list(example.keys())}')

train_dataset = train_dataset.map(convert, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(convert, remove_columns=eval_dataset.column_names)

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        report_to='none',
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        logging_steps=25,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        gradient_checkpointing=True,
        dataloader_num_workers=2,
        seed=42,
    ),
)

stats = trainer.train()
lora_dir = f'{OUTPUT_DIR}/lora'
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)

summary = {
    'train_rows': len(train_dataset),
    'eval_rows': len(eval_dataset),
    'global_step': int(getattr(stats, 'global_step', 0)),
    'training_loss': float(getattr(stats, 'training_loss', 0.0)),
    'lora_dir': lora_dir,
}
print(summary)

if PUSH_TO_HF:
    model.push_to_hub(OUTPUT_REPO, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(OUTPUT_REPO, token=os.environ['HF_TOKEN'])
    print(f'Pushed adapter to {OUTPUT_REPO}')

## Notes

- This lane is code-primary, not game-primary.
- If you add a real C++ corpus later, add it as another train shard and rerun.
- Default is `PUSH_TO_HF = False` so the first run stays local to Colab until you like the result.
